# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook guides you step-by-step in loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Dataset Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

The `mlcroissant` API exposes the Croissant schema entities via the `record_sets` attribute, each with unique `@id` for unambiguous referencing.

In [ ]:
record_sets = dataset.metadata.record_sets

print("Overview of Record Sets and Fields by @id:")
for rs in record_sets:
    print(f"- Record Set: {rs['@id']}")
    fields = rs.get('fields', [])
    for fld in fields:
        print(f"    - Field: {fld['@id']}, Name: {fld.get('name', 'N/A')}, Data Type: {fld.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data for each record set into a Pandas DataFrame. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

# Load records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

# Print information about the first available record set
if dataframes:
    main_rs = list(dataframes.keys())[0]
    print(f"Columns in record set {main_rs}: {dataframes[main_rs].columns.tolist()}")
    dataframes[main_rs].head()
else:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, and categorize or group data by key attributes.

Here, we demonstrate removing outliers, normalizing a numeric column, and grouping by a categorical field, all referenced by their `@id`.

In [ ]:
# Choose a record set for analysis
if dataframes:
    record_set_id = main_rs
    df = dataframes[record_set_id]

    # Attempt to pick a numeric field from the columns
    numeric_field_id = None
    group_field_id = None
    for rs in dataset.metadata.record_sets:
        if rs['@id'] == record_set_id:
            for fld in rs.get('fields', []):
                dt = fld.get('dataType', 'N/A')
                if dt in ['schema:Float', 'schema:Integer', 'schema:Number'] and fld['@id'] in df.columns:
                    numeric_field_id = fld['@id']
                if dt in ['schema:Text', 'schema:DefinedTerm', 'schema:Boolean'] and fld['@id'] in df.columns:
                    group_field_id = fld['@id']

    if numeric_field_id is not None:
        # Filter numeric field > threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if possible
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot the distribution of the selected numeric field, referenced by its `@id`, and visualize grouping if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We explored the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id`. Record sets, fields, and columns were loaded and analyzed in Pandas.

- Record sets and their fields were reviewed via their `@id`.
- Data was extracted, filtered, normalized, and grouped as part of EDA.
- Visualizations were produced for numeric and categorical relationships.

This notebook can be extended for deeper statistical analysis, modeling, or for downstream applications. Use the `@id` references for precise and transparent data handling.